# Hyperparameter search with `PruningSearchCV`

The estimators and their config objects (`TrainingConfig`, `LoRAConfig`, `GenerationConfig`)
subclass `BaseEstimator`, so every nested field is addressable through the usual `__` path and any
scikit-learn-compatible search can tune it. Inside a `Pipeline` step named `lm`:

- `lm__precision` — a flat field on the estimator.
- `lm__training__epochs` — the `epochs` field of the nested `TrainingConfig`.
- `lm__training__lr_scheduler__learning_rate` — one level deeper, on the schedule object.

`sklm.tuning.PruningSearchCV` is an Optuna search built for the cost of fine-tuning. Each trial is
evaluated by $k$-fold cross-validation, and after every fold the running mean is reported to the
study's pruner, so a configuration clearly below the median stops without fine-tuning the remaining
folds. Selection ranks completed trials by $\mathrm{mean} - \lambda\,\mathrm{std}$ over the folds,
preferring a configuration that is consistent across folds.

> With `n_trials=10` and `cv=4` this fine-tunes up to 40 times, fewer once pruning kicks in.

In [ ]:
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from optuna.distributions import CategoricalDistribution, FloatDistribution, IntDistribution
from optuna.logging import WARNING, set_verbosity
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklm import Callback, LanguageModelClassifier
from sklm.tuning import LogCallback, PruningSearchCV, best_trial

set_verbosity(WARNING)  # LogCallback is the view; silence Optuna's own per-trial logs

SEED = 42

## Estimator, pipeline and search space

Fixed knobs live on the estimator; only the swept fields go in `param_distributions`.
`callback=Callback()` keeps every fit silent (the default would attach a dashboard per fit).

In [ ]:
clf = LanguageModelClassifier(
    backend="mlx",
    model="mlx-community/distilgpt2",
    precision="fp32",
    callback=Callback(),
    random_state=SEED,
)
pipe = Pipeline([("lm", clf)])

param_distributions = {
    "lm__training__lr_scheduler__learning_rate": CategoricalDistribution([1e-5, 2e-5, 5e-5]),
    "lm__training__lr_scheduler__warmup_ratio": FloatDistribution(0.0, 0.2),
    "lm__training__epochs": IntDistribution(20, 60),
}

## Run the search

`LogCallback` prints one line per fold and per finished trial, with the risk-adjusted score
$\mathrm{adj} = \mathrm{mean} - \lambda\,\mathrm{std}$ (`std_penalty` is $\lambda$). The winning
configuration is refit on the full training split into `best_estimator_`.

In [ ]:
STD_PENALTY = 0.5

search = PruningSearchCV(
    pipe,
    param_distributions,
    cv=4,
    scoring="accuracy",
    n_trials=10,
    std_penalty=STD_PENALTY,
    sampler=TPESampler(multivariate=True, seed=SEED),
    pruner=MedianPruner(),
    callbacks=[LogCallback(std_penalty=STD_PENALTY)],
    random_state=SEED,
)

iris = load_iris(as_frame=True)
X = iris.data
y = iris.target_names[iris.target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

search.fit(X_train, y_train);

## Best configuration

`best_trial_` is the trial with the highest `mean - λ·std`; ranking the same study with
`std_penalty=0` selects by mean alone, and the two can differ.

In [ ]:
best = search.best_trial_
attrs = best.user_attrs
mean_only = best_trial(search.study_, std_penalty=0.0)
print(
    f"selected trial:   #{best.number}  "
    f"(mean {attrs['mean_test_score']:.3f} +/- {attrs['std_test_score']:.3f}, "
    f"adj {search.best_score_:.3f})"
)
print(f"mean-only best:   #{mean_only.number}")
print(f"best params:      {best.params}")
print(f"test accuracy:    {accuracy_score(y_test, search.predict(X_test)):.3f}")